In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tgt
import nltk
import soundfile as sf
from scipy.fftpack import dct
from tqdm import tqdm
from nltk.corpus import cmudict

# importing the original repo's module for pitch and energy extraction
repo_root = os.getcwd()
utils_path = repo_root / "src" / "utils"
sys.path.insert(0, str(utils_path))

from prosody_tools import f0_processing, energy_processing

# configuration
ROOT_DIR = Path("/Data/Discourse/AUDIO_CHUNKED/Discourse-UWO/FollowUp/")
METADATA_FILE = Path("/Data/Discourse/UWO_splits.csv")
OUTPUT_FILE = Path("/Data/Discourse/prosodic_features_UWO.csv")
CHECKPOINT_FILE = OUTPUT_FILE.with_suffix(".partial.csv")

N_F0_DCT = 4
ENERGY_HZ = 200.0
PITCH_HALF_WINDOW_SEC = 0.125

SAVE_EVERY_N_SPEAKERS = 5

# Cleaning thresholds (e.g. hard cutoffs for implausible acoustic measurements)
MIN_DURATION = 0.03
MAX_DURATION = 3.0
MAX_DURATION_PER_SYLLABLE = 2.0
MAX_PAUSE = 3.0
MIN_MEAN_INTENSITY = 1e-4

WINSOR_LO = 0.005
WINSOR_HI = 0.995
ZSCORE_CLIP = 5.0

# mapping logic for identifying stressed syllables in MFA aligned TextGrids' "phones" tier using CMUdict
nltk.download("cmudict")
cmu = cmudict.dict()

CMU_to_MFA = {
    "AA": {"ɑ", "ɑː", "ɒ", "ɒː", "AA"},
    "AE": {"æ", "æː", "AE"},
    "AH": {"ʌ", "ə", "ɐ", "AH"},
    "AO": {"ɔ", "ɔː", "ɒ", "ɒː", "AO"},
    "AW": {"aʊ", "AW"},
    "AX": {"ə", "AX"},
    "AY": {"aɪ", "æ", "æː", "ə", "AY"},
    "EH": {"ɛ", "EH"},
    "ER": {"ɝ", "ɚ", "əɹ", "ER"},
    "EY": {"eɪ", "EY"},
    "IH": {"ɪ", "ɨ", "i", "IH"},
    "IY": {"i", "iː", "IY"},
    "OW": {"oʊ", "OW"},
    "OY": {"ɔɪ", "OY"},
    "UH": {"ʊ", "UH"},
    "UW": {"u", "uː", "UW"},
}

SILENCE_LABELS = {"", "sil", "sp", "spn", "<sil>", "silence"}


# helpers
def get_valid_wavs_with_textgrids(folder: Path):
    return sorted([p for p in folder.glob("*.wav") if p.with_suffix(".TextGrid").exists()])


def collect_speaker_folders(root_dir: Path):
    return sorted([p for p in root_dir.iterdir() if p.is_dir()])


def read_textgrid_safe(path):
    for enc in ["utf-8", "utf-16", "utf-16-le", "latin-1"]:
        try:
            return tgt.io.read_textgrid(str(path), encoding=enc)
        except Exception:
            continue
    raise ValueError(f"Cannot read {path}")


def count_syllables(word):
    w = str(word).lower()
    if w in cmu:
        pron = cmu[w][0]
        return sum(1 for p in pron if p[-1].isdigit())
    return 1


def normalize_phone(label):
    return str(label).strip().replace("ː", "").lower()


def stressed_syllable_midpoint(word, onset, offset, n_syl, phones_tier):
    try:
        pron = cmu[str(word).lower()][0]

        stressed_idx = None
        for i, p in enumerate(pron):
            if p[-1] == "1":
                stressed_idx = i
                break

        if stressed_idx is None:
            for i, p in enumerate(pron):
                if p[-1].isdigit():
                    stressed_idx = i
                    break

        if stressed_idx is None:
            stressed_idx = 0

        stressed_vowel = pron[stressed_idx][:-1]

        if stressed_vowel in CMU_to_MFA:
            phones = [
                p for p in phones_tier.intervals
                if p.start_time < offset and p.end_time > onset
            ]
            possible = {normalize_phone(x) for x in CMU_to_MFA[stressed_vowel]}

            for p in phones:
                if normalize_phone(p.text) in possible:
                    return (p.start_time + p.end_time) / 2.0
    except Exception:
        pass

    return onset + 0.5 * (offset - onset)


def mean_over_interval(contour, frame_hz, start_sec, end_sec):
    contour = np.asarray(contour, dtype=float)
    i0 = max(0, int(np.floor(start_sec * frame_hz)))
    i1 = min(len(contour), int(np.ceil(end_sec * frame_hz)))

    if i1 <= i0:
        return np.nan

    vals = contour[i0:i1]
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan

    return float(np.mean(vals))


def slice_over_interval(contour, frame_hz, start_sec, end_sec):
    contour = np.asarray(contour, dtype=float)
    i0 = max(0, int(np.floor(start_sec * frame_hz)))
    i1 = min(len(contour), int(np.ceil(end_sec * frame_hz)))

    if i1 <= i0:
        return np.array([], dtype=float)

    vals = contour[i0:i1]
    vals = vals[np.isfinite(vals)]
    return vals


def resample_1d(x, out_len):
    x = np.asarray(x, dtype=float)

    if len(x) == 0:
        return np.full(out_len, np.nan)

    if len(x) == 1:
        return np.full(out_len, x[0], dtype=float)

    xp = np.linspace(0.0, 1.0, len(x))
    xnew = np.linspace(0.0, 1.0, out_len)
    return np.interp(xnew, xp, x)


def dct_vector_from_segment(segment, n_coeffs=4):
    segment = np.asarray(segment, dtype=float)
    segment = segment[np.isfinite(segment)]

    if len(segment) == 0:
        return np.full(n_coeffs, np.nan)

    seg_rs = resample_1d(segment, 32)
    coeffs = dct(seg_rs, type=2, norm="ortho")
    return coeffs[:n_coeffs].astype(float)


def local_prominence_from_contours(logf0_segment, energy_segment, dur_syl):
    vals = []

    logf0_segment = np.asarray(logf0_segment, dtype=float)
    logf0_segment = logf0_segment[np.isfinite(logf0_segment)]
    if len(logf0_segment) > 1:
        vals.append(np.nanmean(np.abs(np.gradient(logf0_segment))))

    energy_segment = np.asarray(energy_segment, dtype=float)
    energy_segment = energy_segment[np.isfinite(energy_segment)]
    if len(energy_segment) > 1:
        vals.append(np.nanmean(np.abs(np.gradient(energy_segment))))

    if np.isfinite(dur_syl):
        vals.append(0.5 * float(dur_syl))

    if len(vals) == 0:
        return np.nan

    return float(np.sum(vals))


def compute_relative_prominence(vals):
    rel = []
    vals = np.asarray(vals, dtype=float)

    for i in range(len(vals)):
        start = max(0, i - 3)

        if i == 0:
            prev = 0.0
        else:
            prev = np.nanmean(vals[start:i])

        rel.append(vals[i] - prev)

    return rel


# pitch contour extraction using repo modules
def extract_repo_style_contours(wav_path: Path):
    waveform, fs = sf.read(str(wav_path))

    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)

    f0_raw = f0_processing.extract_f0(
        waveform,
        fs=fs,
        f0_min=40,
        f0_max=500,
        configuration="pitch_tracker",
    )

    # f0_processing.process internally logs for outlier removal,
    # then returns Hz again if input was raw Hz.
    f0_hz = f0_processing.process(
        f0_raw,
        fix_outliers=True,
        interpolate=True,
    )

    f0_hz = np.asarray(f0_hz, dtype=float)
    f0_hz = np.where(np.isfinite(f0_hz) & (f0_hz > 0), f0_hz, np.nan)

    # we therefore log transform it again to get value distributions alike those in Wolf et al. (2023).
    logf0 = np.log(f0_hz)

    energy = energy_processing.extract_energy(
        waveform,
        fs=fs,
        min_freq=200,
        max_freq=3000,
        method="rms",
        target_rate=200,
    )
    energy_proc = energy_processing.process(energy)
    energy_proc = np.asarray(energy_proc, dtype=float)

    audio_dur = len(waveform) / fs if fs > 0 else 0.0
    f0_hz_rate = len(f0_hz) / audio_dur if audio_dur > 0 and len(f0_hz) > 0 else 200.0

    return waveform, fs, f0_hz, logf0, float(f0_hz_rate), energy_proc


# feature extraction
def process_root():
    rows = []
    speaker_folders = collect_speaker_folders(ROOT_DIR)

    print(f"Found {len(speaker_folders)} speakers")
    print(f"Output will be saved to: {OUTPUT_FILE}")
    print(f"Partial checkpoints will be saved to: {CHECKPOINT_FILE}")

    global_start = time.time()

    for s_i, speaker_dir in enumerate(tqdm(speaker_folders, desc="Speakers"), start=1):
        speaker = speaker_dir.name.strip()
        wav_files = get_valid_wavs_with_textgrids(speaker_dir)

        if len(wav_files) == 0:
            tqdm.write(f"[SKIP] {speaker}: no valid wav/TextGrid pairs")
            continue

        speaker_start = time.time()
        speaker_rows_before = len(rows)

        tqdm.write(f"\n[SPEAKER {s_i}/{len(speaker_folders)}] {speaker} | files={len(wav_files)}")

        for wav_path in tqdm(wav_files, desc=f"{speaker}", leave=False):
            file_start = time.time()
            tg_path = wav_path.with_suffix(".TextGrid")
            utt_id = wav_path.stem

            try:
                _, _, f0_hz, logf0_contour, f0_rate, energy_contour = extract_repo_style_contours(wav_path)
            except Exception as e:
                tqdm.write(f"[WARN] contour extraction failed: {wav_path} | {type(e).__name__}: {e}")
                continue

            try:
                tg = read_textgrid_safe(tg_path)
                words = tg.get_tier_by_name("words")
            except Exception as e:
                tqdm.write(f"[WARN] TextGrid failed: {tg_path} | {type(e).__name__}: {e}")
                continue

            try:
                phones = tg.get_tier_by_name("phones")
            except Exception:
                phones = None

            utter_rows = []
            word_counter = 0

            for idx, w in enumerate(words.intervals):
                word = str(w.text).strip()

                if word.lower() in SILENCE_LABELS:
                    continue

                onset = float(w.start_time)
                offset = float(w.end_time)
                duration = offset - onset

                if duration <= 0:
                    continue

                word_counter += 1

                n_syl = count_syllables(word)
                dur_syl = duration / max(n_syl, 1)

                if idx < len(words.intervals) - 1:
                    next_onset = float(words.intervals[idx + 1].start_time)
                    pause = max(0.0, next_onset - offset)
                else:
                    pause = 0.0

                if phones is not None:
                    stress_time = stressed_syllable_midpoint(word, onset, offset, n_syl, phones)
                else:
                    stress_time = onset + 0.5 * duration

                win_start = max(onset, stress_time - PITCH_HALF_WINDOW_SEC)
                win_end = min(offset, stress_time + PITCH_HALF_WINDOW_SEC)

                # Mean F0 in Hz kept for inspection.
                mean_f0 = mean_over_interval(f0_hz, f0_rate, win_start, win_end)

                # Main pitch features use log-F0.
                mean_logf0 = mean_over_interval(logf0_contour, f0_rate, win_start, win_end)
                logf0_seg = slice_over_interval(logf0_contour, f0_rate, win_start, win_end)
                f0_dct = dct_vector_from_segment(logf0_seg, n_coeffs=N_F0_DCT)

                mean_intensity = mean_over_interval(energy_contour, ENERGY_HZ, onset, offset)
                energy_seg = slice_over_interval(energy_contour, ENERGY_HZ, onset, offset)

                prom_abs = local_prominence_from_contours(logf0_seg, energy_seg, dur_syl)

                row = {
                    "speaker": speaker,
                    "utterance_id": utt_id,
                    "word_index": idx,
                    "word_text": word,
                    "duration": duration,
                    "duration_per_syllable": dur_syl,
                    "pause": pause,
                    "mean_f0": mean_f0,
                    "mean_logf0": mean_logf0,
                    "mean_intensity": mean_intensity,
                    "prom_abs": prom_abs,
                }

                for j in range(N_F0_DCT):
                    row[f"f0_dct_{j}"] = f0_dct[j]

                utter_rows.append(row)

            if utter_rows:
                utt_df = pd.DataFrame(utter_rows).sort_values("word_index")
                rows.extend(utt_df.to_dict("records"))

            file_time = time.time() - file_start
            tqdm.write(f"{speaker}/{utt_id} | words={word_counter} | {file_time:.2f}s")

        speaker_time = time.time() - speaker_start
        speaker_rows = len(rows) - speaker_rows_before

        tqdm.write(
            f"[DONE] {speaker} | files={len(wav_files)} | rows={speaker_rows} | "
            f"time={speaker_time:.1f}s"
        )

        if s_i % SAVE_EVERY_N_SPEAKERS == 0:
            partial_df = pd.DataFrame(rows)
            CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
            partial_df.to_csv(CHECKPOINT_FILE, index=False)
            tqdm.write(f"[CHECKPOINT] saved {len(partial_df)} rows to {CHECKPOINT_FILE}")

    total_time = time.time() - global_start
    print(f"\nTotal extraction time: {total_time / 60:.2f} minutes")

    return pd.DataFrame(rows)


# data cleaning and z-scoring - utilizing configs at start of cell
def winsorize_by_speaker(df, col, lower=WINSOR_LO, upper=WINSOR_HI):
    df = df.copy()
    out = pd.Series(np.nan, index=df.index, dtype=float)

    for spk, idx in df.groupby("speaker").groups.items():
        x = pd.to_numeric(df.loc[idx, col], errors="coerce")
        lo = x.quantile(lower)
        hi = x.quantile(upper)
        out.loc[idx] = x.clip(lo, hi)

    df[col] = out
    return df


def zscore_by_speaker(df, source_col, target_col):
    df = df.copy()
    df[target_col] = np.nan

    for spk, idx in df.groupby("speaker").groups.items():
        x = pd.to_numeric(df.loc[idx, source_col], errors="coerce")
        mean = x.mean(skipna=True)
        std = x.std(skipna=True)

        if not np.isfinite(std) or std < 1e-8:
            df.loc[idx, target_col] = 0.0
        else:
            df.loc[idx, target_col] = (x - mean) / std

    return df


def recompute_prom_rel_group(group):
    group = group.sort_values("word_index").copy()
    group["prom_rel"] = compute_relative_prominence(group["prom_abs"].values)
    return group


def clean_and_normalize_features(df):
    print("\n[CLEANING] Starting cleaning and normalization...")
    df = df.copy()

    n0 = len(df)

    # Hard cutoffs
    df.loc[df["duration"] > MAX_DURATION, "duration"] = np.nan
    df.loc[df["duration_per_syllable"] > MAX_DURATION_PER_SYLLABLE, "duration_per_syllable"] = np.nan
    df.loc[df["pause"] > MAX_PAUSE, "pause"] = np.nan
    df.loc[df["mean_intensity"] < MIN_MEAN_INTENSITY, "mean_intensity"] = np.nan
    df.loc[df["prom_abs"] <= 0, "prom_abs"] = np.nan

    # Lower clips for timing features
    df["duration"] = df["duration"].clip(MIN_DURATION, MAX_DURATION)
    df["duration_per_syllable"] = df["duration_per_syllable"].clip(MIN_DURATION, MAX_DURATION_PER_SYLLABLE)
    df["pause"] = df["pause"].clip(0.0, MAX_PAUSE)

    # Winsorize raw features per speaker
    for col in [
        "mean_logf0",
        "mean_intensity",
        "prom_abs",
        "f0_dct_0",
        "f0_dct_1",
        "f0_dct_2",
        "f0_dct_3",
    ]:
        print(f"[CLEANING] Winsorizing {col}")
        df = winsorize_by_speaker(df, col)

    # Z-score normalized features per speaker
    print("[CLEANING] computing z-scores per speaker")
    df = zscore_by_speaker(df, "mean_logf0", "mean_f0_z")
    df = zscore_by_speaker(df, "mean_intensity", "intensity_z")

    for j in range(N_F0_DCT):
        df = zscore_by_speaker(df, f"f0_dct_{j}", f"f0_dct_{j}_z")

    # Guard against extreme z-scores
    df["mean_f0_z"] = df["mean_f0_z"].clip(-ZSCORE_CLIP, ZSCORE_CLIP)
    df["intensity_z"] = df["intensity_z"].clip(-ZSCORE_CLIP, ZSCORE_CLIP)

    for j in range(N_F0_DCT):
        df[f"f0_dct_{j}_z"] = df[f"f0_dct_{j}_z"].clip(-ZSCORE_CLIP, ZSCORE_CLIP)

    # compute and winsorize relative prominence from absolute prominence
    print("[CLEANING] computing relative prominence")
    df = (
        df.sort_values(["speaker", "utterance_id", "word_index"])
          .groupby(["speaker", "utterance_id"], group_keys=False)
          .apply(recompute_prom_rel_group)
    )

    df["prom_rel"] = df["prom_rel"].clip(
        df["prom_rel"].quantile(0.001),
        df["prom_rel"].quantile(0.999),
    )

    print(f"[CLEANING] Done. Rows: {n0} -> {len(df)}")
    return df


def print_feature_stats(df, title):
    print(f"\n===== {title} =====")
    cols = [
        "duration",
        "duration_per_syllable",
        "pause",
        "mean_f0",
        "mean_logf0",
        "mean_intensity",
        "prom_abs",
        "prom_rel",
        "mean_f0_z",
        "intensity_z",
        "f0_dct_0",
        "f0_dct_1",
        "f0_dct_2",
        "f0_dct_3",
        "f0_dct_0_z",
        "f0_dct_1_z",
        "f0_dct_2_z",
        "f0_dct_3_z",
    ]

    for col in cols:
        if col not in df.columns:
            continue

        x = pd.to_numeric(df[col], errors="coerce")
        print(
            f"{col:24s} "
            f"min={x.min():.4f} "
            f"max={x.max():.4f} "
            f"mean={x.mean():.4f} "
            f"median={x.median():.4f} "
            f"std={x.std():.4f} "
            f"nan={x.isna().sum()}"
        )


# merge with metadata from _splits.csv 
def merge_metadata(df):
    print("\n[METADATA] Merging metadata...")
    meta = pd.read_csv(METADATA_FILE)
    meta.columns = meta.columns.str.strip()

    meta["participant_id"] = meta["participant_id"].astype(str).str.strip()
    df["speaker"] = df["speaker"].astype(str).str.strip()

    meta = meta.set_index("participant_id")

    for col in [
        "gender",
        "dataset",
        "split",
        "label_depression",
        "score_depression",
        "depression_severity",
        "label_psychosis",
        "score_psychosis",
        "psychosis_remission",
    ]:
        if col in meta.columns:
            df[col] = df["speaker"].map(meta[col])
        else:
            print(f"[WARN] Metadata column missing: {col}")
            df[col] = np.nan

    missing = df[df["dataset"].isna()]["speaker"].unique()
    print("Speakers missing metadata:", sorted(missing))

    return df


# run
start = time.time()

df = process_root()

print_feature_stats(df, "RAW EXTRACTED FEATURE STATS")

df = clean_and_normalize_features(df)

print_feature_stats(df, "CLEANED FEATURE STATS")

df = merge_metadata(df)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)

print("\nFeature extraction complete.")
print(f"Saved to: {OUTPUT_FILE}")
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

print(f"\nTotal wall time: {(time.time() - start) / 60:.2f} minutes")

[nltk_data] Downloading package cmudict to /home/ucloud/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


Found 107 speakers
Output will be saved to: /work/DISCOURSE/Data/Discourse/prosodic_features_UWO_repo_style_LOGF0_CLEANED.csv
Partial checkpoints will be saved to: /work/DISCOURSE/Data/Discourse/prosodic_features_UWO_repo_style_LOGF0_CLEANED.partial.csv


Speakers:   0%|          | 0/107 [00:00<?, ?it/s]


[SPEAKER 1/107] 001 | files=47



Speakers:   0%|          | 0/107 [00:00<?, ?it/s]

...

148/148_042 | words=25 | 0.65s



Speakers: 100%|██████████| 107/107 [1:38:49<00:00, 55.42s/it]


148/148_043 | words=36 | 1.04s
[DONE] 148 | files=43 | rows=1051 | time=38.9s

Total extraction time: 98.83 minutes

===== RAW EXTRACTED FEATURE STATS =====
duration                 min=0.0100 max=14.5500 mean=0.2881 median=0.2400 std=0.2503 nan=0
duration_per_syllable    min=0.0060 max=14.5500 mean=0.2372 median=0.1900 std=0.2263 nan=0
pause                    min=0.0000 max=19.3900 mean=0.0855 median=0.0000 std=0.3010 nan=0
mean_f0                  min=40.0000 max=498.8515 mean=174.5958 median=178.7147 std=75.2168 nan=0
mean_logf0               min=3.6889 max=6.2123 mean=5.0522 median=5.1798 std=0.4780 nan=0
mean_intensity           min=0.0005 max=0.4546 mean=0.0842 median=0.0737 std=0.0482 nan=0
prom_abs                 min=0.0051 max=7.2893 mean=0.1323 median=0.1091 std=0.1128 nan=0
f0_dct_0                 min=20.8675 max=35.1425 mean=28.5779 median=29.3001 std=2.7034 nan=0
f0_dct_1                 min=-3.6536 max=4.8739 mean=0.0010 median=0.0088 std=0.5909 nan=0
f0_dct_2         

/tmp/ipykernel_512574/3819375796.py:510: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(recompute_prom_rel_group)


[CLEANING] Done. Rows: 160240 -> 160240

===== CLEANED FEATURE STATS =====
duration                 min=0.0300 max=3.0000 mean=0.2848 median=0.2400 std=0.2160 nan=129
duration_per_syllable    min=0.0300 max=2.0000 mean=0.2315 median=0.1900 std=0.1734 nan=312
pause                    min=0.0000 max=3.0000 mean=0.0820 median=0.0000 std=0.2652 nan=130
mean_f0                  min=40.0000 max=498.8515 mean=174.5958 median=178.7147 std=75.2168 nan=0
mean_logf0               min=3.6889 max=6.2084 mean=5.0521 median=5.1798 std=0.4767 nan=0
mean_intensity           min=0.0038 max=0.3585 mean=0.0841 median=0.0737 std=0.0475 nan=0
prom_abs                 min=0.0108 max=1.9485 mean=0.1307 median=0.1091 std=0.0939 nan=0
prom_rel                 min=-0.4795 max=0.8565 mean=0.0052 median=-0.0052 std=0.1073 nan=0
mean_f0_z                min=-5.0000 max=4.3337 mean=0.0008 median=0.0210 std=0.9953 nan=0
intensity_z              min=-2.1702 max=5.0000 mean=-0.0001 median=-0.2156 std=0.9993 nan=0
f0_dc